In [121]:
import torch.nn as nn
import torch
import os
from torch.utils.data import Dataset, DataLoader
import pickle
import numpy as np
import math
from analysis import Logger, AverageMeter, accuracy, binary_accuracy
from torchvision import transforms
from torch.utils.data import BatchSampler
from torch.utils.data import Dataset, DataLoader
from PIL import Image

LOG_FILE = "log_incomplete.txt"
BEST_MODEL_NAME = "best_model_incomplete"
PKL_FILE_DIR = "class_attr_data_10_updated_random_removed_concepts/"
ARGS_N_ATTRIBUTES = 56 # number of attributes to use for training (if using subset of attributes, then set this to the number of attributes used, otherwise set to N_ATTRIBUTES)

DATA_DIR = "Data/"
LOG_DIR = "Log/"

N_ATTRIBUTES = 312
N_CLASSES = 200

WEIGHTED_LOSS = False
MIN_LR = 0.0001
LR = 0.001
LR_DECAY_SIZE = 0.1
N_CLASS_ATTR = 2 # whether attr prediction is a binary or triary classification
EXPAND_DIM = 0 # dimension of hidden layer (if we want to increase model capacity) - for bottleneck only (DEFAULT 0, i.e. no expansion)
NO_IMG = True # if included, only use attributes (and not raw imgs) for class prediction
USE_ATTR = True # whether to use attributes (FOR COTRAINING ARCHITECTURE ONLY)
OPTIMIZER = 'SGD' # Type of optimizer to use, options incl SGD, RMSProp, Adam
CKPT = "" # For retraining on both train + val set
BATCH_SIZE = 64
SCHEDULER_STEP = 100 # Number of steps before decaying current learning rate by half
ATTR_LOSS_WEIGHT = 1.0 # weight for loss by predicting attributes
BOTTLENECK = False # whether to predict attributes before class labels
WEIGHT_DECAY = 0.00005 # weight decay for optimizer
UNCERTAIN_LABELS = False # whether to use (normalized) attribute certainties as labels (If used then true)
EPOCHS = 200 # epochs for training process
IMAGE_DIR = "images" # default is images, test image folder to run inference on
RESAMPLING = False # whether to resample training data to have balanced attribute distribution (if used then true)
SAVE_STEP = 1000 # Default = 1000. Number of epochs between saving model checkpoints


NORMALIZE_LOSS = False # Whether to normalize loss by taking attr_loss_weight into account (if used then true)
USE_AUX = False # Whether to use auxiliary loss for predicting attributes in addition to main loss for predicting class labels (if used then true)
SEED = 1



In [122]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:    
    device = torch.device('cpu')

In [123]:
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes, expand_dim):
        super(MLP, self).__init__()
        self.expand_dim = expand_dim
        if self.expand_dim:
            self.linear = nn.Linear(input_dim, expand_dim)
            self.activation = torch.nn.ReLU()
            self.linear2 = nn.Linear(expand_dim, num_classes) #softmax is automatically handled by loss function
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        x = self.linear(x)
        if hasattr(self, 'expand_dim') and self.expand_dim:
            x = self.activation(x)
            x = self.linear2(x)
        return x


In [124]:
class CUBDataset(torch.utils.data.Dataset):
    def __init__(self, pkl_file_paths, use_attr, no_img, uncertain_label, image_dir, n_class_attr, transform=None):
        """
        Arguments:
        pkl_file_paths: list of full path to all the pkl data
        use_attr: whether to load the attributes (e.g. False for simple finetune)
        no_img: whether to load the images (e.g. False for A -> Y model)
        uncertain_label: if True, use 'uncertain_attribute_label' field (i.e. label weighted by uncertainty score, e.g. 1 & 3(probably) -> 0.75)
        image_dir: default = 'images'. Will be append to the parent dir
        n_class_attr: number of classes to predict for each attribute. If 3, then make a separate class for not visible
        transform: whether to apply any special transformation. Default = None, i.e. use standard ImageNet preprocessing
        """
        # Load data from appropriate pkl files
        self.data = []
        self.is_train = any(["train" in path for path in pkl_file_paths])
        if not self.is_train:
            assert any([("test" in path) or ("val" in path) for path in pkl_file_paths])
        for file_path in pkl_file_paths:
            self.data.extend(pickle.load(open(file_path, 'rb')))
        self.transform = transform
        self.use_attr = use_attr
        self.no_img = no_img
        self.uncertain_label = uncertain_label
        self.image_dir = image_dir
        self.n_class_attr = n_class_attr
        
    def __len__(self):
        return len(self.data)
    
    
    def __getitem__(self, idx):
        img_data = self.data[idx]
        img_path = img_data['img_path']
        try:
            img = Image.open(img_path).convert('RGB')
        except:
            print(f"Error loading image at path: {img_path}")
            raise

        class_label = img_data['class_label']
        # Trandform image if necessary
        if self.transform:
            img = self.transform(img)

        if self.use_attr:
            if self.uncertain_label:
                attr_label = img_data['uncertain_attribute_label']
            else:
                attr_label = img_data['attribute_label']
            if self.no_img:
                if self.n_class_attr == 3:
                    one_hot_attr_label = np.zeros((N_ATTRIBUTES, self.n_class_attr))
                    one_hot_attr_label[np.arange(N_ATTRIBUTES), attr_label] = 1
                    return one_hot_attr_label, class_label
                else:
                    # Only return attribute labels and class labels (c to y model)
                    return attr_label, class_label
            else:
                # Return image, attribute labels and class labels
                return img, class_label, attr_label
        else:
            # Return image class labels (x to y model)
            return img, class_label

        
            
            
            

In [125]:
class ImbalancedDatasetSampler(torch.utils.data.sampler.Sampler):
    """Samples elements randomly from a given list of indices for imbalanced dataset
    Arguments:
        indices (list, optional): a list of indices
        num_samples (int, optional): number of samples to draw
    """

    def __init__(self, dataset, indices=None):
        # if indices is not provided,
        # all elements in the dataset will be considered
        self.indices = list(range(len(dataset))) \
            if indices is None else indices

        # if num_samples is not provided,
        # draw `len(indices)` samples in each iteration
        self.num_samples = len(self.indices)

        # distribution of classes in the dataset
        label_to_count = {}
        for idx in self.indices:
            label = self._get_label(dataset, idx)
            if label in label_to_count:
                label_to_count[label] += 1
            else:
                label_to_count[label] = 1

        # weight for each sample
        weights = [1.0 / label_to_count[self._get_label(dataset, idx)]
                   for idx in self.indices]
        self.weights = torch.DoubleTensor(weights)

    def _get_label(self, dataset, idx):  # Note: for single attribute dataset
        return dataset.data[idx]['attribute_label'][0]

    def __iter__(self):
        idx = (self.indices[i] for i in torch.multinomial(
            self.weights, self.num_samples, replacement=True))
        return idx

    def __len__(self):
        return self.num_samples

In [126]:
def load_data(pkl_paths, use_attr, no_img, batch_size, uncertain_label=False, n_class_attr=2, image_dir='images', resampling=False, resol=299):
    """
    Note: Inception needs (299,299,3) images with inputs scaled between -1 and 1
    Loads data with transformations applied, and upsample the minority class if there is class imbalance and weighted loss is not used
    NOTE: resampling is customized for first attribute only, so change sampler.py if necessary
    """
    resized_resol = int(resol * 256/224)
    is_training = any(['train.pkl' in f for f in pkl_paths])
    if is_training:
        transform = transforms.Compose([
            #transforms.Resize((resized_resol, resized_resol)),
            #transforms.RandomSizedCrop(resol),
            transforms.ColorJitter(brightness=32/255, saturation=(0.5, 1.5)),
            transforms.RandomResizedCrop(resol),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(), #implicitly divides by 255
            transforms.Normalize(mean = [0.5, 0.5, 0.5], std = [2, 2, 2])
            #transforms.Normalize(mean = [ 0.485, 0.456, 0.406 ], std = [ 0.229, 0.224, 0.225 ]),
            ])
    else:
        transform = transforms.Compose([
            #transforms.Resize((resized_resol, resized_resol)),
            transforms.CenterCrop(resol),
            transforms.ToTensor(), #implicitly divides by 255
            transforms.Normalize(mean = [0.5, 0.5, 0.5], std = [2, 2, 2])
            #transforms.Normalize(mean = [ 0.485, 0.456, 0.406 ], std = [ 0.229, 0.224, 0.225 ]),
            ])

    dataset = CUBDataset(pkl_paths, use_attr, no_img, uncertain_label, image_dir, n_class_attr, transform)
    if is_training:
        drop_last = True
        shuffle = True
    else:
        drop_last = False
        shuffle = False
    if resampling:
        sampler = BatchSampler(ImbalancedDatasetSampler(dataset), batch_size=batch_size, drop_last=drop_last)
        loader = DataLoader(dataset, batch_sampler=sampler)
    else:
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)
    return loader

In [127]:
# Calculated total number of samples / number of positive samples for each attribute (or overall) to determine imbalance ratio for weighted loss = n/n_ones - 1
# 100 samples, 10 ones and 90 zeros => imbalance ratio = 100/10 - 1 = 9.0, so weight positive samples 9x more than negative samples in loss function
def find_class_imbalance(pkl_file, multiple_attr=False, attr_idx=-1):
    """
    Calculate class imbalance ratio for binary attribute labels stored in pkl_file
    If attr_idx >= 0, then only return ratio for the corresponding attribute id
    If multiple_attr is True, then return imbalance ratio separately for each attribute. Else, calculate the overall imbalance across all attributes
    """
    imbalance_ratio = []
    data = pickle.load(open(DATA_DIR + pkl_file, 'rb'))
    n = len(data)
    n_attr = len(data[0]['attribute_label'])
    if attr_idx >= 0:
        n_attr = 1
    if multiple_attr:
        n_ones = [0] * n_attr
        total = [n] * n_attr
    else:
        n_ones = [0]
        total = [n * n_attr]
    for d in data:
        labels = d['attribute_label']
        if multiple_attr:
            for i in range(n_attr):
                n_ones[i] += labels[i]
        else:
            if attr_idx >= 0:
                n_ones[0] += labels[attr_idx]
            else:
                n_ones[0] += sum(labels)
    for j in range(len(n_ones)):
        imbalance_ratio.append(total[j]/n_ones[j] - 1)
    if not multiple_attr: #e.g. [9.0] --> [9.0] * 312
        imbalance_ratio *= n_attr
    return imbalance_ratio


In [128]:
def run_epoch_simple(model, optimizer, loader, loss_meter, acc_meter, criterion, is_training):
    """
    A -> Y: Predicting class labels using only attributes with MLP
    """
    if is_training:
        model.train()
    else:
        model.eval()
    for _, data in enumerate(loader):
        inputs, labels = data
        if isinstance(inputs, list):
            #inputs = [i.long() for i in inputs]
            inputs = torch.stack(inputs).t().float()
        inputs = torch.flatten(inputs, start_dim=1).float()
        inputs_var = torch.autograd.Variable(inputs).to(device)
        inputs_var = inputs_var.to(device) 
        labels_var = torch.autograd.Variable(labels).to(device)
        labels_var = labels_var.to(device)
        outputs = model(inputs_var)
        loss = criterion(outputs, labels_var)
        acc = accuracy(outputs, labels, topk=(1,))
        loss_meter.update(loss.item(), inputs.size(0))
        acc_meter.update(acc[0], inputs.size(0))

        if is_training:
            optimizer.zero_grad() #zero the parameter gradients
            loss.backward()
            optimizer.step() #optimizer step to update parameters
    return loss_meter, acc_meter

In [129]:
def run_epoch(model, optimizer, loader, loss_meter, acc_meter, criterion, attr_criterion, is_training):
    """
    For the rest of the networks (X -> A, cotraining, simple finetune)
    """
    if is_training:
        model.train()
    else:
        model.eval()

    for _, data in enumerate(loader):
        if attr_criterion is None:
            inputs, labels = data
            attr_labels, attr_labels_var = None, None
        else:
            inputs, labels, attr_labels = data
            if ARGS_N_ATTRIBUTES > 1:
                attr_labels = [i.long() for i in attr_labels]
                attr_labels = torch.stack(attr_labels).t()#.float() #N x 312
            else:
                if isinstance(attr_labels, list):
                    attr_labels = attr_labels[0]
                attr_labels = attr_labels.unsqueeze(1)
            attr_labels_var = torch.autograd.Variable(attr_labels).float()
            attr_labels_var = attr_labels_var.to(device) 

        inputs_var = torch.autograd.Variable(inputs)
        inputs_var = inputs_var.to(device) 
        labels_var = torch.autograd.Variable(labels)
        labels_var = labels_var.to(device) 

        if is_training and USE_AUX:
            outputs, aux_outputs = model(inputs_var)
            losses = []
            out_start = 0
            if not BOTTLENECK: #loss main is for the main task label (always the first output)
                loss_main = 1.0 * criterion(outputs[0], labels_var) + 0.4 * criterion(aux_outputs[0], labels_var)
                losses.append(loss_main)
                out_start = 1
            if attr_criterion is not None and ATTR_LOSS_WEIGHT > 0: #X -> A, cotraining, end2end
                for i in range(len(attr_criterion)):
                    losses.append(ATTR_LOSS_WEIGHT * (1.0 * attr_criterion[i](outputs[i+out_start].squeeze().float(), attr_labels_var[:, i]) \
                                                            + 0.4 * attr_criterion[i](aux_outputs[i+out_start].squeeze().float(), attr_labels_var[:, i])))
        else: #testing or no aux logits
            outputs = model(inputs_var)
            losses = []
            out_start = 0
            if not BOTTLENECK:
                loss_main = criterion(outputs[0], labels_var)
                losses.append(loss_main)
                out_start = 1
            if attr_criterion is not None and ATTR_LOSS_WEIGHT > 0: #X -> A, cotraining, end2end
                for i in range(len(attr_criterion)):
                    losses.append(ATTR_LOSS_WEIGHT * attr_criterion[i](outputs[i+out_start].squeeze().float(), attr_labels_var[:, i]))

        if BOTTLENECK: #attribute accuracy
            sigmoid_outputs = torch.nn.Sigmoid()(torch.cat(outputs, dim=1))
            acc = binary_accuracy(sigmoid_outputs, attr_labels)
            acc_meter.update(acc.data.cpu().numpy(), inputs.size(0))
        else:
            acc = accuracy(outputs[0], labels, topk=(1,)) #only care about class prediction accuracy
            acc_meter.update(acc[0], inputs.size(0))

        if attr_criterion is not None:
            if BOTTLENECK:
                total_loss = sum(losses)/ ARGS_N_ATTRIBUTES
            else: #cotraining, loss by class prediction and loss by attribute prediction have the same weight
                total_loss = losses[0] + sum(losses[1:])
                if NORMALIZE_LOSS:
                    total_loss = total_loss / (1 + ATTR_LOSS_WEIGHT * ARGS_N_ATTRIBUTES)
        else: #finetune
            total_loss = sum(losses)
        loss_meter.update(total_loss.item(), inputs.size(0))
        if is_training:
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
    return loss_meter, acc_meter


In [130]:
def train(model):
    # Determine imbalance
    imbalance = None
    if USE_ATTR and not NO_IMG and WEIGHTED_LOSS:
        train_data_path = os.path.join(DATA_DIR + PKL_FILE_DIR + 'train.pkl')
        if WEIGHTED_LOSS == 'multiple':
            imbalance = find_class_imbalance(train_data_path, True)
        else:
            imbalance = find_class_imbalance(train_data_path, False)

    # Commented out because do not want ot delete old logs for now, but can uncomment if want to start fresh log for new experiments
    """
    if os.path.exists(LOG_DIR): # job restarted by cluster
        for f in os.listdir(LOG_DIR):
            os.remove(os.path.join(LOG_DIR, f))
    else:
        os.makedirs(LOG_DIR)
    """

    logger = Logger(os.path.join(LOG_DIR, LOG_FILE))
    #logger.write(str(args) + '\n')
    logger.write(str(imbalance) + '\n')
    logger.flush()

    model = model.to(device)
    criterion = torch.nn.CrossEntropyLoss()
    if USE_ATTR and not NO_IMG:
        attr_criterion = [] #separate criterion (loss function) for each attribute
        if WEIGHTED_LOSS:
            assert(imbalance is not None)
            for ratio in imbalance:
                attr_criterion.append(torch.nn.BCEWithLogitsLoss(weight=torch.FloatTensor([ratio]).to(device)))
        else:
            for i in range(ARGS_N_ATTRIBUTES):
                attr_criterion.append(torch.nn.CrossEntropyLoss())
    else:
        attr_criterion = None

    if OPTIMIZER == 'Adam':
        optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)
    elif OPTIMIZER == 'RMSprop':
        optimizer = torch.optim.RMSprop(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, momentum=0.9, weight_decay=WEIGHT_DECAY)
    else:
        optimizer = torch.optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, momentum=0.9, weight_decay=WEIGHT_DECAY)
    #scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.1, patience=5, threshold=0.00001, min_lr=0.00001, eps=1e-08)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=SCHEDULER_STEP, gamma=0.1)
    stop_epoch = int(math.log(MIN_LR / LR) / math.log(LR_DECAY_SIZE)) * SCHEDULER_STEP
    print("Stop epoch: ", stop_epoch)

    train_data_path = os.path.join(DATA_DIR + PKL_FILE_DIR + 'train.pkl')
    val_data_path = train_data_path.replace('train.pkl', 'val.pkl')
    logger.write('train data path: %s\n' % train_data_path)

    # Load data. If retraining, then train and val sets are combined for training and we only evaluate on the train set, since the model should be able to reach 100% accuracy on both train and val
    if CKPT: #retraining
        train_loader = load_data([train_data_path, val_data_path], USE_ATTR, NO_IMG, BATCH_SIZE, UNCERTAIN_LABELS, image_dir=IMAGE_DIR, \
                                 n_class_attr=N_CLASS_ATTR, resampling=RESAMPLING)
        val_loader = None
    else:
        train_loader = load_data([train_data_path], USE_ATTR, NO_IMG, BATCH_SIZE, UNCERTAIN_LABELS, image_dir=IMAGE_DIR, \
                                 n_class_attr=N_CLASS_ATTR, resampling=RESAMPLING)
        val_loader = load_data([val_data_path], USE_ATTR, NO_IMG, BATCH_SIZE, image_dir=IMAGE_DIR, n_class_attr=N_CLASS_ATTR)

    best_val_epoch = -1
    best_val_loss = float('inf')
    best_val_acc = 0

    for epoch in range(0, EPOCHS):
        train_loss_meter = AverageMeter()
        train_acc_meter = AverageMeter()
        if NO_IMG:
            train_loss_meter, train_acc_meter = run_epoch_simple(model, optimizer, train_loader, train_loss_meter, train_acc_meter, criterion, is_training=True)
        else:
            train_loss_meter, train_acc_meter = run_epoch(model, optimizer, train_loader, train_loss_meter, train_acc_meter, criterion, attr_criterion, is_training=True)
 
        if not CKPT: # evaluate on val set
            val_loss_meter = AverageMeter()
            val_acc_meter = AverageMeter()
        
            with torch.no_grad():
                if NO_IMG:
                    val_loss_meter, val_acc_meter = run_epoch_simple(model, optimizer, val_loader, val_loss_meter, val_acc_meter, criterion, is_training=False)
                else:
                    val_loss_meter, val_acc_meter = run_epoch(model, optimizer, val_loader, val_loss_meter, val_acc_meter, criterion, attr_criterion, is_training=False)

        else: #retraining
            val_loss_meter = train_loss_meter
            val_acc_meter = train_acc_meter

        if best_val_acc < val_acc_meter.avg:
            best_val_epoch = epoch
            best_val_acc = val_acc_meter.avg
            logger.write('New model best model at epoch %d\n' % epoch)
            torch.save(model, os.path.join(LOG_DIR, BEST_MODEL_NAME + '_%d.pth' % SEED))
            #if best_val_acc >= 100: #in the case of retraining, stop when the model reaches 100% accuracy on both train + val sets
            #    break

        train_loss_avg = train_loss_meter.avg
        val_loss_avg = val_loss_meter.avg
        
        logger.write('Epoch [%d]:\tTrain loss: %.4f\tTrain accuracy: %.4f\t'
                'Val loss: %.4f\tVal acc: %.4f\t'
                'Best val epoch: %d\n'
                % (epoch, train_loss_avg, train_acc_meter.avg, val_loss_avg, val_acc_meter.avg, best_val_epoch)) 
        logger.flush()
        
        if epoch <= stop_epoch:
            scheduler.step(epoch) #scheduler step to update lr at the end of epoch     
        #inspect lr
        if epoch % 10 == 0:
            print('Current lr:', scheduler.get_lr())

        # if epoch % args.save_step == 0:
        #     torch.save(model, os.path.join(LOG_DIR, '%d_model.pth' % epoch))

        if epoch >= 100 and val_acc_meter.avg < 3:
            print("Early stopping because of low accuracy")
            break
        if epoch - best_val_epoch >= 100:
            print("Early stopping because acc hasn't improved for a long time")
            break


In [131]:
# Safely close the log file
def safe_close(self):
    if self.file is not None and not self.file.closed:
        self.file.close()

Logger.close = safe_close
Logger.__del__ = safe_close

In [132]:
# Independent Model
def ModelOracleCtoY(n_class_attr, n_attributes, num_classes, expand_dim):
    # X -> C part is separate, this is only the C -> Y part
    if n_class_attr == 3:
        model = MLP(input_dim=n_attributes * n_class_attr, num_classes=num_classes, expand_dim=expand_dim)
    else:
        model = MLP(input_dim=n_attributes, num_classes=num_classes, expand_dim=expand_dim)
    return model

model = ModelOracleCtoY(n_class_attr=N_CLASS_ATTR, n_attributes=ARGS_N_ATTRIBUTES,
                        num_classes=N_CLASSES, expand_dim=EXPAND_DIM)




train(model)
# need to evaluate model

None
Stop epoch:  100
train data path: Data/class_attr_data_10_updated_random_removed_concepts/train.pkl
New model best model at epoch 0
Epoch [0]:	Train loss: 5.3170	Train accuracy: 0.9924	Val loss: 5.2999	Val acc: 0.9182	Best val epoch: 0
Current lr: [0.001]


/var/folders/wb/9whmmh7s5bvf174ppr2jwprr0000gn/T/ipykernel_11504/1639790825.py:108: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  scheduler.step(epoch) #scheduler step to update lr at the end of epoch


Epoch [1]:	Train loss: 5.2884	Train accuracy: 1.6047	Val loss: 5.2730	Val acc: 0.9182	Best val epoch: 0
New model best model at epoch 2
Epoch [2]:	Train loss: 5.2602	Train accuracy: 1.8370	Val loss: 5.2466	Val acc: 1.2521	Best val epoch: 2
Epoch [3]:	Train loss: 5.2319	Train accuracy: 2.1748	Val loss: 5.2204	Val acc: 1.2521	Best val epoch: 2
Epoch [4]:	Train loss: 5.2036	Train accuracy: 2.2171	Val loss: 5.1942	Val acc: 1.2521	Best val epoch: 2
Epoch [5]:	Train loss: 5.1758	Train accuracy: 2.1748	Val loss: 5.1680	Val acc: 1.2521	Best val epoch: 2
New model best model at epoch 6
Epoch [6]:	Train loss: 5.1483	Train accuracy: 2.5338	Val loss: 5.1421	Val acc: 2.0033	Best val epoch: 6
New model best model at epoch 7
Epoch [7]:	Train loss: 5.1207	Train accuracy: 4.4764	Val loss: 5.1163	Val acc: 4.0067	Best val epoch: 7
New model best model at epoch 8
Epoch [8]:	Train loss: 5.0934	Train accuracy: 6.1233	Val loss: 5.0908	Val acc: 5.6761	Best val epoch: 8
New model best model at epoch 9
Epoch [9